# Real-Gas Molecular Dynamics
### A discovery-driven, AI-collaborative lab

In this notebook you will use molecular-dynamics simulations of noble gases to
*rediscover* two cornerstones of real-gas behavior:

1. How a real gas approaches the ideal-gas law in the dilute limit, and
2. Why different gases look different at the same temperature, yet become
   *the same* when described in reduced units (the [law of corresponding states](https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Physical_Chemistry_(LibreTexts)/16%3A_The_Properties_of_Gases/16.04%3A_The_Law_of_Corresponding_States)).
3. Visualize real gas simulations using [directly in a web browser](https://jschrier.github.io/MolecularDynamics/)

You will work **with an AI assistant** (Colab's built-in Gemini, or any chat model).
The rule of thumb throughout: **you decide and interpret; the AI writes code; the
package does the computing.** The simulations run for free on this machine — the AI is
only ever used to help you *set up* and *make sense of* experiments, never inside the
simulation loop.

## Learning objectives

By the end of this notebook you should be able to:

- Explain the compressibility factor $Z = \dfrac{PV}{Nk_BT}$ and what $Z=1$ means.
- Run a parameter *sweep* over density at fixed temperature and plot $Z$ vs. density.
- Verify that $Z \to 1$ and $PV/nT \to R$ in the dilute limit, and use that limit as a
  sanity check for judging whether a simulation (or an AI's code) is trustworthy.
- Compare several noble gases and connect their differing non-ideality to the depth of
  the Lennard-Jones well.
- Transform to **reduced units** ($T^{*} = T/(\varepsilon/k_B)$, $\rho^{*} = n\sigma^3$)
  and demonstrate the **law of corresponding states** by collapsing all gases onto one
  curve.
- Practice **predict-then-test**: commit to a prediction *before* running.
- Practice **responsible AI use**: read the code the assistant writes, and check its
  output against a physical properties you trust before believing it.

## How this notebook works

Each activity has four parts:

- 🔮 **Predict** — write down what you expect *before* running anything. The discovery is
  in the gap between your prediction and the result.
- 🤖 **Starter prompt** — a prompt you can paste into Colab's Gemini panel (or another
  chat model) to generate the code. Treat what it returns as a *draft*: read it, run it,
  and compare against the reference cell.
- ▶️ **Run & check** — execute the code and look at the result.
- 🔎 **Interpret** — answer the questions in your own words.

> **Cost note.** Every simulation here runs locally and is free. The only metered resource
> is the AI assistant, and you only need it a handful of times (to draft each code cell).
> Keep AI calls *out* of loops — draft the code once, then let it run.

## Setup

Install the package and import what we need. (On Colab the install takes a few seconds.)

In [1]:
%pip install noblegasmd -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import noblegasmd as md

print("noblegasmd version:", md.__version__)
print("available gases:", list(md.GAS_CONSTANTS.keys()))

Note: you may need to restart the kernel to use updated packages.
noblegasmd version: 0.1.0
available gases: ['He', 'Ne', 'Ar', 'Kr', 'Xe']


### Test with one simulation

`md.run(...)` runs a single simulation and returns a result object. A few things to notice:
the pressure is measured from **wall collisions** (kinetic-theory definition), so it needs
the full trajectory to build up — very short runs report `Z = 0` simply because too few
collisions have been sampled. Use the default length for anything involving pressure or $Z$.

In [2]:
r = md.run(gas="Ar", T=300.0, rho=40.0, seed=1)   # default n_steps = 20000
print(f"gas={r.gas}  T_avg={r.T_avg:.1f} K   Z={r.Z:.4f}   PV/nT={r.gc:.3f} (R=8.314)")
print(f"energy drift over the run = {r.energy_drift:.2e}   <- NVE: should be tiny")

gas=Ar  T_avg=299.1 K   Z=0.9955   PV/nT=8.277 (R=8.314)
energy drift over the run = 1.15e-05   <- NVE: should be tiny


---
# Goal 1 — Guided reproduction (the AI as scribe)

**Goal:** recover the ideal-gas limit for a single gas, and learn the sweep-and-plot
workflow on ground you can check.

### Background
The **compressibility factor** $Z = \dfrac{PV}{Nk_BT}$ measures how far a gas departs from
ideal behavior: $Z = 1$ is an ideal gas exactly. Equivalently $PV/nT = R \approx 8.314\
\mathrm{J\,mol^{-1}K^{-1}}$ for an ideal gas. A real gas approaches these values as it
becomes dilute (low density), because the particles spend almost all their time far apart
where the Lennard-Jones forces are negligible.

`md.sweep(...)` runs many simulations over a grid of conditions and returns a tidy
`pandas.DataFrame` — one row per (temperature, density, replicate). Useful columns:
`T_set`, `rho_set` (the requested conditions), `Z`, `gc` (that's $PV/nT$), and `seed`.

### 🔮 Predict (do this before running)

> **Anchor: what does "number density" feel like?**
>
> An ideal gas at STP (0 °C, 1 bar) has
> $$\frac{n}{V} = \frac{P}{RT} = \frac{10^5~\text{Pa}}{(8.314)(273.15)} \approx 44~\text{mol/m}^3$$
>
> Since $1~\text{m}^3 = 1000~\text{L}$, that's $0.044~\text{mol/L}$ — or the familiar
> molar volume $22.7~\text{L/mol}$. (Reminder: $1~\text{mol/L} = 1000~\text{mol/m}^3$.)
>
> So on the axis below:
> - **~40 mol/m³** ≈ ordinary atmospheric pressure
> - **~1000 mol/m³** ≈ 25 bar at 300 K
> - **~2000–4000 mol/m³** ≈ 50–100 bar — a compressed gas cylinder

For **argon at 300 K**, sketch or describe how you expect $Z$ to behave as number density
increases from very low (a few tens of mol/m³) to a few thousand mol/m³.

- At the **lowest** density, what value should $Z$ be near? Why?
- Does $Z$ rise above 1, dip below 1, or stay flat as density increases — and what would
  each of those mean physically?

*Your prediction:*

> _(write here)_

### 🤖 Starter prompt

Paste something like this into the Gemini panel to draft the code, then compare with the
reference cell below:

> *I have a Python package `noblegasmd` (imported as `md`) already installed. It provides
> `md.sweep(gas, T, rho, n_replicates, seed)` which returns a pandas DataFrame with columns
> including `rho_set`, `Z`, and `gc`. Write code that runs **argon at T = 300 K** across
> number densities from 40 to 5000 mol/m³ (about 8 points) with 3 replicate seeds each.
> Plot the mean $Z$ vs. density with error bars from the replicate spread, and draw a dashed
> line at Z = 1. Label axes with units.*

Then **read the code it gives you.** Does it actually average the replicates? Does it use
the default run length (needed for meaningful pressure)? Run it, then check it against the
reference.

### ▶️ Your cell — paste / adapt the AI's code here

### 🔎 Interpret

1. Does $Z \to 1$ (and $PV/nT \to R$) at the lowest density? Within the error bars?
2. At what density does $Z$ start to deviate noticeably from 1, and in which direction?
3. The error bars come from running the same conditions with different random seeds. Why are they larger at the lowest densities? *(Hint: what physically produces the pressure, and how often does it happen when the box is nearly empty?)*
4. Open the [Browswer-enabled MD Simulation Tool](https://jschrier.github.io/MolecularDynamics/) to run and visualize the simulation with (a) the smallest density, and (b) the smallest $Z$ value.
5.  In both simulations, try to visually observe the motion of a single particle until it collides with another particle or with the wall.  After the first collision, repeat this observation two more times with different particles
6.  Which simulation, (a) or (b), do particle collisions appear more frequently?  **Hint** you can correlate the time between collisions with the frame count.  
7.  In the simulation in which particles collide more frequently, what behavior do you observe as two particles near collision?



**Responsible-AI check:** if the assistant's code had a bug, which single feature of this
   plot would most quickly tell you something was wrong?



---
# Goal 2 — Parameter exploration (the AI as lab partner)

**Goal:** discover *why* gases differ, and then discover that the differences largely
disappear in the right units.

### Background
Each noble gas has its own Lennard-Jones parameters: a well depth $\varepsilon$ (how
strongly two atoms attract) and a size $\sigma$. Heavier noble gases have deeper wells.
The package stores these as per-gas conversion factors, which you can read:

- `md.get_gas_constants(gas).temp_fac` is $\varepsilon/k_B$ in **kelvin**
- `md.get_gas_constants(gas).vol_fac` is $\sigma^3$ in **m³**

That lets you convert any real condition into **reduced units**:

$$ T^{*} = \frac{T\,[\mathrm{K}]}{\texttt{temp\_fac}}, \qquad
   \rho^{*} = n\,\sigma^3 = \rho\,[\mathrm{mol/m^3}]\times N_A \times \texttt{vol\_fac}. $$

The **law of corresponding states** claims that, written in these reduced units, *all*
Lennard-Jones gases obey the *same* equation of state — $Z$ is a universal function of
$(T^{*}, \rho^{*})$.

In [ ]:
# Peek at the per-gas constants so the reduced-unit transforms aren't a black box.
for g in ["He", "Ne", "Ar", "Kr", "Xe"]:
    gc = md.get_gas_constants(g)
    print(f"{g}:  eps/kB = {gc.temp_fac:8.3f} K     sigma^3 = {gc.vol_fac:.3e} m^3")

### 🔮 Predict (do this before running)

**Part A — same real temperature.** You will simulate all five noble gases at the *same*
real temperature, 300 K, and plot $Z$ vs. density.

- Which gas do you expect to deviate **most** from ideal ($Z=1$)? Which **least**? Why?
  *(Hint: relate 300 K to each gas's well depth $\varepsilon/k_B$ printed above.)*

**Part B — same reduced temperature.** Then you will re-run each gas at the same *reduced*
temperature $T^{*}$ and plot $Z$ vs. $\rho^{*}$.

- What do you predict happens to the five curves in reduced units?

> _(your predictions)_

### 🤖 Starter prompt — Part A (compare gases at 300 K)

> *Using `md.sweep` (returns a DataFrame with `gas`, `rho_set`, `Z`), run all five noble
> gases `["He","Ne","Ar","Kr","Xe"]` at T = 300 K over number densities 40–5000 mol/m³
> (about 8 points, 2 replicates). Plot mean $Z$ vs. density as one line per gas on the same
> axes, with a dashed line at Z = 1.*

### ▶️ Your cell — Part A

In [ ]:
# Your (AI-drafted) Part A code here.


### Reference — Part A *(peek after trying)*

In [ ]:
# --- Tier 2A reference: five gases at the SAME real temperature ---
gases = ["He", "Ne", "Ar", "Kr", "Xe"]
densities = np.linspace(40, 5000, 8)

frames = [md.sweep(gas=g, T=[300.0], rho=densities, n_replicates=2, seed=0) for g in gases]
dfA = pd.concat(frames, ignore_index=True)

plt.figure(figsize=(6, 4))
for g in gases:
    sub = dfA[dfA.gas == g].groupby("rho_set")["Z"].mean()
    plt.plot(sub.index, sub.values, marker="o", label=g)
plt.axhline(1.0, ls="--", color="gray")
plt.xlabel("number density  (mol/m$^3$)")
plt.ylabel("Z")
plt.title("Five noble gases at the SAME real temperature (300 K)")
plt.legend()
plt.show()

### 🤖 Starter prompt — Part B (collapse in reduced units)

This is the subtle, important step. To test corresponding states you must compare the gases
at the **same reduced temperature** $T^{*}$ — which means a *different real* temperature for
each gas.

> *For each gas in `["He","Ne","Ar","Kr","Xe"]`, pick a target reduced temperature
> `Tstar = 2.0`. Using `md.get_gas_constants(gas)`, compute the real temperature
> `T = Tstar * temp_fac` and, for a shared grid of reduced densities `rho_star` from 0.02 to
> 0.40, the real densities `rho = rho_star / (md.NA * vol_fac)`. Run `md.sweep` for each gas
> at its real T and densities (2 replicates), convert back to `rho_star`, and plot mean $Z$
> vs. `rho_star` as one line per gas.*

### ▶️ Your cell — Part B

In [ ]:
# Your (AI-drafted) Part B code here.


### Reference — Part B *(peek after trying)*

In [ ]:
# --- Tier 2B reference: same REDUCED temperature -> corresponding states ---
Tstar = 2.0                                   # supercritical: smooth, no phase transition
rho_star_grid = np.linspace(0.02, 0.40, 8)    # reduced density (must stay well below 1)

frames = []
for g in gases:
    gc = md.get_gas_constants(g)
    T_real   = Tstar * gc.temp_fac                       # K
    rho_real = rho_star_grid / (md.NA * gc.vol_fac)      # mol/m^3
    d = md.sweep(gas=g, T=[T_real], rho=rho_real, n_replicates=2, seed=0)
    d["rho_star"] = d["rho_set"] * md.NA * gc.vol_fac
    frames.append(d)
dfB = pd.concat(frames, ignore_index=True)

plt.figure(figsize=(6, 4))
for g in gases:
    sub = dfB[dfB.gas == g].groupby("rho_star")["Z"].mean()
    plt.plot(sub.index, sub.values, marker="o", label=g)
plt.axhline(1.0, ls="--", color="gray")
plt.xlabel(r"reduced density  $\rho^* = n\sigma^3$")
plt.ylabel("Z")
plt.title(f"All five gases at the SAME reduced temperature  T* = {Tstar}")
plt.legend()
plt.show()

### 🔎 Interpret

1. In **Part A**, which gas deviated most from ideal at 300 K, and which least? Does the
   ordering match your prediction and the well depths $\varepsilon/k_B$?
2. In **Part B**, did the five curves collapse onto (nearly) one? What does that imply — that
   at equal $T^{*}$ and $\rho^{*}$, helium and xenon are essentially *the same gas*?
3. The real temperatures in Part B ranged from ~22 K (He) to ~560 K (Xe). Why is it
   physically reasonable that such different conditions give the same $Z$?
4. Where the collapse is *imperfect*, what might be responsible? (Consider quantum effects
   for helium, statistical noise, or the finite number of particles.)

> _(answers here)_

---
### 💡 Going further (Tier 3 preview)

The low-density slope of $Z$ vs. $\rho$ is not just noise around 1 — its leading term gives
the **second virial coefficient** $B_2(T)$, the first correction to ideality. Sweeping $T$
and finding where $B_2$ changes sign locates the **Boyle temperature** (near $T^{*} \approx
3.4$ for a Lennard-Jones fluid). That's the next tier: extracting a real equation-of-state
coefficient from a pair potential.

---
## Rubric

Each criterion is scored 0–3. (Instructors: adjust weights to taste.)

| Criterion | 3 — Exemplary | 2 — Proficient | 1 — Developing | 0 — Missing |
|---|---|---|---|---|
| **Predictions** | Specific, physically reasoned predictions recorded *before* running, for both tiers | Predictions recorded, partial reasoning | Vague or only one tier | None recorded |
| **Simulation setup** | Correct sweeps at full run length; sensible density/temperature grids; replicates used | Mostly correct; minor grid or length issues | Runs but with a flaw (e.g., too-short runs → Z = 0) | Non-functional |
| **Plots** | Clear, labeled, units correct; ideal-gas reference and error bars shown | Readable with minor omissions | Present but unclear/mislabeled | Missing |
| **Tier 1 interpretation** | Correctly connects dilute limit to $Z\to1$, $PV/nT\to R$; explains onset of deviation | Correct limit, thin explanation | Partly correct | Absent/incorrect |
| **Tier 2 interpretation** | Explains the collapse as corresponding states; links deviation to well depth | Identifies the collapse, limited explanation | Notices a pattern only | Absent/incorrect |
| **Responsible AI use** | Reads/verifies AI code; checks output against an invariant (Z→1, energy drift); catches or rules out errors | Some verification | Runs AI code unchecked but sensibly | Blind copy-paste, errors uncaught |

**Total: /18**

---
### Appendix — notes for instructors

- **Timing.** A full run is ~5 s on a typical CPU; free Colab may be 2–3× slower. Each
  reference cell runs `n_densities × n_replicates` (× `n_gases` in Tier 2) simulations —
  budget accordingly, and have students shrink grids to iterate quickly.
- **The `Z = 0` trap.** Pressure is sampled from wall collisions, so runs that are too short
  (or densities that are extremely low) report `Z = 0`. This is a *feature* for teaching
  sampling, but warn students to use the default length for pressure/$Z$.
- **Keep the AI out of loops.** The design uses the assistant only to draft each cell. If a
  student puts an AI/API call inside a sweep, they'll burn quota fast for no benefit — flag
  this explicitly.
- **Corresponding states needs equal $T^{*}$.** The common mistake is comparing gases at equal
  *real* temperature and expecting a collapse. Part A vs. Part B is built to surface exactly
  that distinction.
- The live browser simulation remains the better tool for *watching* the dynamics; this
  notebook is for quantitative sweeps.